In [9]:
import json
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split

In [10]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Using device:", device)

Using device: cuda


In [11]:
images_path = Path("dataset/images")
data = json.loads(Path("dataset/gt_public.json").read_text())
image_ids = list(data.keys())

train_ids, rest_ids = train_test_split(image_ids, test_size=0.2, random_state=0)
val_ids, test_ids = train_test_split(rest_ids, test_size=0.5, random_state=0)

print(len(train_ids), len(val_ids), len(test_ids))

41470 5184 5184


In [12]:
def make_mask(boxes):
    mask = np.zeros((256, 256), dtype=np.uint8)
    yy, xx = np.mgrid[0:256, 0:256]
    for x, y, w, h in boxes:
        cx = x + w / 2
        cy = y + h / 2
        r = max(w, h) / 2
        inside = (xx + 0.5 - cx) ** 2 + (yy + 0.5 - cy) ** 2 <= r ** 2
        mask[inside] = 1
    return mask

In [13]:
import transformers

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(transformers.__version__)

cuda
5.22.1


In [14]:
from huggingface_hub import login

login()

In [15]:
from transformers import AutoModel

model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
dino = AutoModel.from_pretrained(model_name).to(device)
dino.eval()

print(sum(p.numel() for p in dino.parameters()))

W1102 08:55:22.409000 23932 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 211/211 [00:00<00:00, 5274.85it/s]

21596544


In [16]:
from PIL import Image

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def load_image(image_id):
    image = Image.open(images_path / f"{image_id}.png").convert("RGB")
    image = torch.tensor(np.array(image), dtype=torch.float32).permute(2, 0, 1) / 255
    image = (image - mean) / std
    return image

In [17]:
image = load_image(train_ids[0]).unsqueeze(0).to(device)

with torch.no_grad():
    output = dino(pixel_values=image)

print(image.shape)
print(output.last_hidden_state.shape)
print(dino.config.num_register_tokens)

torch.Size([1, 3, 256, 256])
torch.Size([1, 261, 384])
4


In [18]:
from tqdm import tqdm

skip = 1 + dino.config.num_register_tokens

def get_features(ids, batch_size=32):
    all_features = []
    for start in tqdm(range(0, len(ids), batch_size), mininterval=1.0):
        batch_ids = ids[start:start + batch_size]
        images = torch.stack([load_image(i) for i in batch_ids]).to(device)

        with torch.no_grad():
            output = dino(pixel_values=images)

        patches = output.last_hidden_state[:, skip:, :]
        patches = patches.reshape(-1, 16, 16, 384).permute(0, 3, 1, 2)
        all_features.append(patches.half().cpu())

    return torch.cat(all_features)

In [19]:
train_features = get_features(train_ids)
val_features = get_features(val_ids)
test_features = get_features(test_ids)

print(train_features.shape, val_features.shape, test_features.shape)

100%|██████████| 162/162 [00:10<00:00, 16.12it/s]

torch.Size([41470, 384, 16, 16]) torch.Size([5184, 384, 16, 16]) torch.Size([5184, 384, 16, 16])


In [20]:
from torch.utils.data import Dataset, DataLoader

class FeatureDataset(Dataset):
    def __init__(self, features, ids):
        self.features = features
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        feature = self.features[i].float()
        mask = make_mask(data[self.ids[i]])
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return feature, mask

In [21]:
train_loader = DataLoader(FeatureDataset(train_features, train_ids), batch_size=64, shuffle=True)
val_loader = DataLoader(FeatureDataset(val_features, val_ids), batch_size=64)

features, masks = next(iter(train_loader))
print(features.shape, masks.shape)

torch.Size([64, 384, 16, 16]) torch.Size([64, 1, 256, 256])


In [22]:
import torch.nn as nn
import torch.nn.functional as F

class DinoHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.out = nn.Conv2d(384, 1, kernel_size=1)

    def forward(self, x):
        x = self.out(x)
        x = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)
        return x

In [23]:
model = DinoHead().to(device)
print(sum(p.numel() for p in model.parameters()))

with torch.no_grad():
    out = model(features.to(device))
print(out.shape)

385
torch.Size([64, 1, 256, 256])


In [24]:
pos_weight = torch.tensor([10.0]).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

epochs = 15
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [25]:
cutoffs = [0.5, 0.6, 0.7, 0.8, 0.9]

def evaluate(loader):
    model.eval()
    total_loss = 0
    intersection = {c: 0 for c in cutoffs}
    union = {c: 0 for c in cutoffs}

    with torch.no_grad():
        for features, masks in loader:
            features = features.to(device)
            masks = masks.to(device)

            logits = model(features)
            total_loss += loss_fn(logits, masks).item()

            probs = torch.sigmoid(logits)
            for c in cutoffs:
                preds = (probs > c).float()
                intersection[c] += (preds * masks).sum().item()
                union[c] += ((preds + masks) > 0).sum().item()

    ious = {c: intersection[c] / union[c] for c in cutoffs}
    best_cutoff = max(ious, key=ious.get)
    return total_loss / len(loader), ious[best_cutoff], best_cutoff

In [26]:
print(evaluate(val_loader))

(0.9003875844272566, 0.024970969605972233, 0.5)


In [26]:
import time

run_dir = Path("src/dinov3/01-vits16-frozen-linear-head-cosine-15epochs")
run_dir.mkdir(parents=True, exist_ok=True)

history = {"train_loss": [], "val_loss": [], "val_iou": [], "best_cutoff": [], "lr": []}
best_iou = 0

for epoch in range(epochs):
    start = time.time()
    model.train()
    total_loss = 0

    for features, masks in tqdm(train_loader, mininterval=1.0):
        features = features.to(device)
        masks = masks.to(device)

        logits = model(features)
        loss = loss_fn(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    train_loss = total_loss / len(train_loader)
    val_loss, val_iou, best_cutoff = evaluate(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)
    history["best_cutoff"].append(best_cutoff)
    history["lr"].append(lr)
    (run_dir / "history.json").write_text(json.dumps(history, indent=2))

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), run_dir / "model.pt")

    print(f"epoch {epoch + 1}: train {train_loss:.4f}, val {val_loss:.4f}, "
          f"IoU {val_iou:.4f} @ {best_cutoff}, {time.time() - start:.0f}s")

100%|██████████| 648/648 [00:43<00:00, 15.06it/s]


epoch 1: train 0.3519, val 0.2788, IoU 0.4458 @ 0.7, 48s


100%|██████████| 648/648 [00:35<00:00, 18.08it/s]


epoch 2: train 0.2551, val 0.2542, IoU 0.4733 @ 0.8, 40s


100%|██████████| 648/648 [00:36<00:00, 17.84it/s]


epoch 3: train 0.2405, val 0.2448, IoU 0.4823 @ 0.8, 41s


... (20 outputs left out)

100%|██████████| 648/648 [00:36<00:00, 17.86it/s]


epoch 14: train 0.2217, val 0.2304, IoU 0.4973 @ 0.8, 41s


100%|██████████| 648/648 [00:36<00:00, 17.86it/s]


epoch 15: train 0.2216, val 0.2303, IoU 0.4971 @ 0.8, 41s


In [27]:
import matplotlib.pyplot as plt

# 1. Notes
best_epoch = history["val_iou"].index(max(history["val_iou"]))
best_cutoff = history["best_cutoff"][best_epoch]

notes = f"""model: frozen DINOv3 ViT-S/16 (21,596,544 parameters, not trained) + linear head (1x1 conv, 384 -> 1), 385 trained parameters
features: 16x16 patch tokens (384 numbers each), extracted once; head output upsampled to 256x256 (bilinear)
loss: BCEWithLogitsLoss, pos_weight=10
optimizer: Adam, lr=1e-3, cosine schedule, no weight decay
batch size: 64
epochs: {epochs}
cutoffs tried: {cutoffs}
best epoch: {best_epoch + 1}
best val IoU: {history["val_iou"][best_epoch]:.4f} at cutoff {best_cutoff}
"""

(run_dir / "notes.txt").write_text(notes)

# 2. Loss, IoU, and learning rate curves
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

ax1.plot(epochs_range, history["train_loss"], label="Train")
ax1.plot(epochs_range, history["val_loss"], label="Validation")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(epochs_range, history["val_iou"], label="Validation")
ax2.axvline(best_epoch + 1, linestyle="--", color="gray", label="Best epoch")
ax2.set_title("IoU (best cutoff per epoch)")
ax2.set_xlabel("Epoch")
ax2.legend()

ax3.plot(epochs_range, history["lr"])
ax3.set_title("Learning rate")
ax3.set_xlabel("Epoch")

plt.tight_layout()
plt.savefig(run_dir / "curves.png", dpi=150, bbox_inches="tight")
plt.show()

# 3. Example predictions from the best saved model
model.load_state_dict(torch.load(run_dir / "model.pt", map_location=device))
model.eval()

fig, axes = plt.subplots(4, 3, figsize=(9, 12))

for i, row in enumerate(axes):
    image = Image.open(images_path / f"{val_ids[i]}.png").convert("L")
    mask = make_mask(data[val_ids[i]])

    with torch.no_grad():
        pred = torch.sigmoid(model(val_features[i:i + 1].float().to(device)))
    pred = (pred > best_cutoff).float().squeeze().cpu()

    row[0].imshow(image, cmap="gray")
    row[0].set_title(f"Image {val_ids[i]}")
    row[1].imshow(mask, cmap="gray")
    row[1].set_title("True mask")
    row[2].imshow(pred, cmap="gray")
    row[2].set_title(f"Predicted (cutoff {best_cutoff})")
    for ax in row:
        ax.axis("off")

plt.tight_layout()
plt.savefig(run_dir / "predictions.png", dpi=150, bbox_inches="tight")
plt.show()

NameError: name 'history' is not defined

In [29]:
   model = DinoHead().to(device)
   run_dir = Path("src/dinov3/01-baseline")

In [30]:
from scipy import ndimage
from torchvision.ops import box_iou

# same matching rule as Faster R-CNN: a predicted box counts if box IoU >= 0.5 with an unused true crater
def count_matches(pred_boxes, true_boxes, iou_threshold=0.5):
    if len(pred_boxes) == 0 or len(true_boxes) == 0:
        return 0

    ious = box_iou(pred_boxes, true_boxes)
    matches = 0
    for i in range(len(pred_boxes)):
        j = ious[i].argmax()
        if ious[i, j] >= iou_threshold:
            matches += 1
            ious[:, j] = -1  # this real crater is taken
    return matches


# each separate blob in the predicted mask becomes one box (x1, y1, x2, y2)
def mask_to_boxes(pred_mask):
    labeled, n = ndimage.label(pred_mask)
    boxes = []
    for ys, xs in ndimage.find_objects(labeled):
        boxes.append([xs.start, ys.start, xs.stop, ys.stop])
    return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)


crater_cutoffs = [0.5, 0.6, 0.7, 0.8, 0.9]
tp = {c: 0 for c in crater_cutoffs}
n_pred = {c: 0 for c in crater_cutoffs}
n_true = 0

# best saved head
model.load_state_dict(torch.load(run_dir / "model.pt", map_location=device))
model.eval()

for start in tqdm(range(0, len(val_ids), 64), mininterval=1.0):
    with torch.no_grad():
        probs = torch.sigmoid(model(val_features[start:start + 64].float().to(device)))
    probs = probs.squeeze(1).cpu().numpy()

    for prob, image_id in zip(probs, val_ids[start:start + 64]):
        # true boxes, converted the same way as in Faster R-CNN
        true_boxes = [[x, y, x + w, y + h] for x, y, w, h in data[image_id] if w > 0 and h > 0]
        true_boxes = torch.tensor(true_boxes, dtype=torch.float32).reshape(-1, 4)
        n_true += len(true_boxes)

        for c in crater_cutoffs:
            pred_boxes = mask_to_boxes(prob > c)
            n_pred[c] += len(pred_boxes)
            tp[c] += count_matches(pred_boxes, true_boxes)

# precision, recall, F1 for every cutoff, same format as the U-Net's crater_scores.json
scores = {}
for c in crater_cutoffs:
    precision = tp[c] / n_pred[c] if n_pred[c] > 0 else 0.0
    recall = tp[c] / n_true
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    scores[str(c)] = {"precision": precision, "recall": recall, "f1": f1}

(run_dir / "crater_scores.json").write_text(json.dumps(scores))

for c, s in scores.items():
    print(f"cutoff {c}: precision {s['precision']:.4f}, recall {s['recall']:.4f}, F1 {s['f1']:.4f}")

100%|██████████| 81/81 [00:10<00:00,  7.79it/s]

cutoff 0.5: precision 0.0881, recall 0.1278, F1 0.1043
cutoff 0.6: precision 0.1246, recall 0.1519, F1 0.1369
cutoff 0.7: precision 0.1733, recall 0.1728, F1 0.1730
cutoff 0.8: precision 0.2345, recall 0.1817, F1 0.2048
cutoff 0.9: precision 0.3112, recall 0.1634, F1 0.2143
